In [ ]:
from pathlib import Path


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in (current, *current.parents):
        if (path / "data").exists():
            return path
    return current


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"

import os

import numpy as np
import pandas as pd
import tushare as ts

TOKEN = os.getenv("TUSHARE_TOKEN", "").strip()
HTTP_URL = os.getenv("TUSHARE_HTTP_URL", "").strip()
OUT_DIR = DATA_ROOT / "data_download" / "tusahre_day_download" / "stock_day"
START_DATE = "20150101"
END_DATE = "20270101"
LIMIT = 6000

FIELDS_DAILY = "ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount"
FIELDS_ADJ = "ts_code,trade_date,adj_factor"
FIELDS_BASIC = "ts_code,trade_date,close,turnover_rate,turnover_rate_f,volume_ratio,pe,pe_ttm,pb,ps,ps_ttm,dv_ratio,dv_ttm,total_share,float_share,free_share,total_mv,circ_mv"
FIELDS_LIMIT = "ts_code,trade_date,up_limit,down_limit"
FIELDS_ST = "ts_code,trade_date"

if not TOKEN:
    raise RuntimeError("TUSHARE_TOKEN is required")

pro = ts.pro_api(TOKEN)
if HTTP_URL:
    pro._DataApi__http_url = HTTP_URL

OUT_DIR.mkdir(parents=True, exist_ok=True)


def paged_fetch(fetch_fn, limit=LIMIT, **kwargs):
    offset = 0
    parts = []
    while True:
        frame = fetch_fn(limit=limit, offset=offset, **kwargs)
        if frame is None or frame.empty:
            break
        parts.append(frame)
        if len(frame) < limit:
            break
        offset += limit
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def get_open_trade_dates(start_date, end_date):
    calendar = paged_fetch(
        pro.trade_cal,
        exchange="",
        start_date=start_date,
        end_date=end_date,
        fields="cal_date,is_open",
    )
    if calendar.empty:
        raise RuntimeError("trade calendar is empty")
    calendar["is_open"] = pd.to_numeric(calendar["is_open"], errors="coerce").fillna(0).astype(int)
    dates = calendar.loc[calendar["is_open"] == 1, "cal_date"].astype(str).tolist()
    return sorted(dates)


def rounded_numeric(series):
    return pd.to_numeric(series, errors="coerce").round(2)


def merge_optional_daily_table(daily, frame, columns, on):
    if frame is None or frame.empty:
        for column in columns:
            if column not in daily.columns:
                daily[column] = pd.NA
        return daily
    frame = frame.drop_duplicates(subset=on, keep="last")
    return daily.merge(frame[on + columns], on=on, how="left")


def process_one_day(trade_date):
    daily = paged_fetch(pro.daily, trade_date=trade_date, fields=FIELDS_DAILY)
    if daily.empty:
        print(f"warn daily_empty date={trade_date}")
        return 0

    adj = paged_fetch(pro.adj_factor, trade_date=trade_date, fields=FIELDS_ADJ)
    daily = merge_optional_daily_table(daily, adj, ["adj_factor"], ["ts_code"])

    basic = paged_fetch(pro.daily_basic, trade_date=trade_date, fields=FIELDS_BASIC)
    if basic is not None and not basic.empty and "close" in basic.columns:
        basic = basic.drop(columns=["close"])
    if basic is not None and not basic.empty:
        daily = daily.merge(basic, on=["ts_code", "trade_date"], how="left")

    try:
        limit_frame = paged_fetch(pro.stk_limit, trade_date=trade_date, fields=FIELDS_LIMIT)
    except Exception as exc:
        print(f"warn stk_limit_failed date={trade_date} error={repr(exc)}")
        limit_frame = pd.DataFrame()
    daily = merge_optional_daily_table(daily, limit_frame, ["up_limit", "down_limit"], ["ts_code"])

    try:
        st_frame = paged_fetch(pro.stock_st, trade_date=trade_date, fields=FIELDS_ST, limit=1000)
    except Exception as exc:
        print(f"warn stock_st_failed date={trade_date} error={repr(exc)}")
        st_frame = pd.DataFrame()
    if st_frame is not None and not st_frame.empty:
        st_frame = st_frame[["ts_code"]].drop_duplicates()
        st_frame["is_st"] = True
        daily = daily.merge(st_frame, on="ts_code", how="left")
        daily["is_st"] = daily["is_st"].fillna(False)
    else:
        daily["is_st"] = False

    for column in ["open", "close", "high", "low", "up_limit", "down_limit", "vol", "amount"]:
        daily[column] = pd.to_numeric(daily[column], errors="coerce")

    daily["vwap"] = np.where(
        daily["vol"].notna() & (daily["vol"] != 0) & daily["amount"].notna(),
        daily["amount"] * 10.0 / daily["vol"],
        np.nan,
    )
    daily["is_limit_up_close"] = rounded_numeric(daily["close"]) == rounded_numeric(daily["up_limit"])
    daily["is_limit_down_close"] = rounded_numeric(daily["close"]) == rounded_numeric(daily["down_limit"])
    daily["hit_limit_up"] = rounded_numeric(daily["high"]) >= rounded_numeric(daily["up_limit"])
    daily["hit_limit_down"] = rounded_numeric(daily["low"]) <= rounded_numeric(daily["down_limit"])
    daily["is_limit_up_open"] = rounded_numeric(daily["open"]) == rounded_numeric(daily["up_limit"])
    daily["is_limit_down_open"] = rounded_numeric(daily["open"]) == rounded_numeric(daily["down_limit"])

    daily = daily.rename(columns={"ts_code": "code", "trade_date": "date"})
    out_path = OUT_DIR / f"{trade_date}.parquet"
    daily.to_parquet(out_path, index=False)
    return len(daily)


def run(start_date, end_date):
    trade_dates = get_open_trade_dates(start_date, end_date)
    print(f"plan days={len(trade_dates)} start={trade_dates[0]} end={trade_dates[-1]}")
    for trade_date in trade_dates:
        rows = process_one_day(trade_date)
        print(f"saved date={trade_date} rows={rows}")
    print(f"done output={OUT_DIR}")


if __name__ == "__main__":
    run(START_DATE, END_DATE)